# 04 - Optimization Improvements

## Purpose

Document and apply targeted Day 5 improvements.

This notebook focuses on small, explainable improvements that make the project cleaner, easier to debug, or more efficient.

## Required improvements

At least three improvements should be documented.

## Main idea

The goal is visible engineering improvement, not perfect performance.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"

In [0]:
optimization_improvements = [
    Row(
        improvement_id="OPT-001",
        area="Repeated actions",
        original_pattern="Calling df.count() multiple times in the same notebook.",
        improved_pattern="Store count results in variables and reuse them for logs and validation.",
        why_it_helps="Avoids triggering repeated Spark jobs for the same DataFrame.",
        applied_to="Validation and inventory notebooks",
        tradeoff="Counts are still Spark actions, but they are executed fewer times.",
    ),
    Row(
        improvement_id="OPT-002",
        area="Column pruning before joins",
        original_pattern="Joining DataFrames while carrying many unused columns.",
        improved_pattern="Select only business-relevant columns before joins.",
        why_it_helps="Reduces schema width, avoids duplicate columns, and makes joins easier to debug.",
        applied_to="Integrated Silver join and executive risk dashboard view",
        tradeoff="Requires being explicit about which columns are needed.",
    ),
    Row(
        improvement_id="OPT-003",
        area="Stable view source",
        original_pattern="Creating a persistent Unity Catalog view from a temporary view.",
        improved_pattern="Write a managed Delta base table first, then create the persistent view from that table.",
        why_it_helps="Makes the view reusable and not dependent on a notebook session.",
        applied_to="Executive energy risk dashboard view",
        tradeoff="Adds one extra managed table, but improves reliability.",
    ),
    Row(
        improvement_id="OPT-004",
        area="Validation after writes",
        original_pattern="Writing output tables without immediate structured validation.",
        improved_pattern="Validate row counts, key nulls, duplicate grains, and sample outputs after writes.",
        why_it_helps="Catches issues close to where they are introduced.",
        applied_to="Silver, Gold, and Night Shift outputs",
        tradeoff="Adds notebook cells, but improves trust and presentation evidence.",
    ),
    Row(
        improvement_id="OPT-005",
        area="Explicit logging",
        original_pattern="Notebooks relying only on implicit execution output.",
        improved_pattern="Print source tables, target tables, row counts, grain, and business logic summaries.",
        why_it_helps="Makes debugging and review easier after execution.",
        applied_to="Day 4 and Day 5 notebooks",
        tradeoff="More notebook output, but more explainable runs.",
    ),
]

optimization_improvements_df = spark.createDataFrame(optimization_improvements)

display(optimization_improvements_df)

In [0]:
summary_df = (
    optimization_improvements_df
    .groupBy("area")
    .count()
    .orderBy("area")
)

display(summary_df)

In [0]:
dashboard_table = f"{catalog}.analytics.gold_regional_operations_dashboard"

dashboard_df = spark.table(dashboard_table)

# Better pattern: calculate once, reuse many times
dashboard_row_count = dashboard_df.count()

print("Dashboard table:", dashboard_table)
print("Rows:", dashboard_row_count)
print("Rows reused for validation:", dashboard_row_count)

if dashboard_row_count <= 0:
    raise ValueError("Dashboard table is empty.")

print("Repeated count improvement demonstrated.")

In [0]:
market_df = spark.table(f"{catalog}.analytics.gold_daily_market_summary")
weather_df = spark.table(f"{catalog}.analytics.gold_weather_impact_summary")

market_pruned_df = market_df.select(
    "report_day",
    "region",
    "avg_price_eur_mwh",
    "total_volume_mwh",
    "is_high_market_price",
)

weather_pruned_df = weather_df.select(
    "report_day",
    "region",
    "avg_wind_speed_kmh",
    "weather_risk_signal",
)

market_weather_review_df = (
    market_pruned_df
    .join(
        weather_pruned_df,
        on=["report_day", "region"],
        how="left"
    )
)

display(market_weather_review_df)

print("Column pruning before join demonstrated.")
print("Market columns before pruning:", len(market_df.columns))
print("Market columns after pruning:", len(market_pruned_df.columns))
print("Weather columns before pruning:", len(weather_df.columns))
print("Weather columns after pruning:", len(weather_pruned_df.columns))

In [0]:
duplicate_grain_count = (
    market_weather_review_df
    .groupBy("report_day", "region")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate day-region rows:", duplicate_grain_count)

if duplicate_grain_count > 0:
    raise ValueError("Optimization review failed: duplicate day-region grain found.")

print("Duplicate grain validation demonstrated.")

In [0]:
print("Optimization improvements completed.")
print("At least three targeted improvements are documented and demonstrated.")
print("The project is cleaner, easier to debug, and more presentation-ready.")